# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gulgumusdere/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [8]:
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

# Colab secret'tan HF token'ı al (w03'te yaptığın gibi)
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [9]:
feature_df = con.sql("""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr,
        gsc_sum_position * 1.0 / NULLIF(gsc_impressions, 0) AS avg_position
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions >= 50
),
tiered AS (
    SELECT *,
        CASE
            WHEN avg_position <= 3 THEN 'top_3'
            WHEN avg_position <= 10 THEN 'page_1'
            WHEN avg_position <= 20 THEN 'page_2'
            WHEN avg_position <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS position_tier
    FROM base
),
tier_medians AS (
    SELECT position_tier, MEDIAN(ctr) AS expected_ctr
    FROM tiered
    GROUP BY position_tier
)
SELECT
    t.*,
    m.expected_ctr,
    m.expected_ctr - t.ctr AS ctr_gap
FROM tiered t
JOIN tier_medians m USING (position_tier)
""").df()

print(feature_df.shape)
feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(1037442, 11)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,ctr,avg_position,position_tier,expected_ctr,ctr_gap
0,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,2026-03-01,321,0,2005,0.000,6.246106,page_1,0.0,0.000
1,client_62f4a7e64f5e0096,content_26f5092ee7f70d45,2026-03-01,87,0,550,0.000,6.321839,page_1,0.0,0.000
2,client_62f4a7e64f5e0096,content_9e7c70abfbae371e,2026-03-01,139,0,814,0.000,5.856115,page_1,0.0,0.000
3,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,2026-03-01,125,2,403,0.016,3.224000,page_1,0.0,-0.016
4,client_62f4a7e64f5e0096,content_85b1be9944e4e19d,2026-03-01,50,0,356,0.000,7.120000,page_1,0.0,0.000


## 2. Feature notes (meaning, missing, categorical, available-when?)

**Feature notes**

- **client_hash_id, content_hash_id**: identifiers, pseudonymized. No missing values. Available at prediction time (they define the row).

- **report_date**: the month window (2026-03). Available at prediction time.

- **gsc_impressions, gsc_clicks, gsc_sum_position**: raw GSC metrics for the month — already-closed monthly totals, not a live/streaming count. If predicting *during* March before month-end, these would not be fully available yet. Treating month=2026-03 as a closed historical window here — noted as a limitation in section 4. No missing values after the `gsc_impressions >= 50` filter.

- **ctr, avg_position**: derived directly from the metrics above. Same availability caveat.

- **position_tier**: derived from avg_position via fixed bins I chose (top_3, page_1, page_2, page_3_5, deep). Available at prediction time — deterministic function of avg_position.

- **expected_ctr**: MEDIAN(ctr) computed per position_tier, across the *same rows being scored* — a leakage risk, addressed in section 3.

- **ctr_gap**: expected_ctr − ctr. This is the proxy label (target), never a model input.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [11]:
# Test 1: Does any planned model INPUT overlap with the label itself?
label_col = 'ctr_gap'
candidate_inputs = ['gsc_impressions', 'position_tier', 'avg_position',
                     'client_hash_id', 'content_hash_id']

# ctr and expected_ctr are explicitly EXCLUDED from candidate_inputs
# because ctr_gap = expected_ctr - ctr. Using either as a feature
# would let the model see its own label.
print("Label:", label_col)
print("Planned inputs:", candidate_inputs)
print("Excluded (label components): ctr, expected_ctr")


Label: ctr_gap
Planned inputs: ['gsc_impressions', 'position_tier', 'avg_position', 'client_hash_id', 'content_hash_id']
Excluded (label components): ctr, expected_ctr


In [12]:
# Leakage found: expected_ctr's MEDIAN was computed over the SAME rows
# being scored, so each row's own ctr slightly influences its own baseline.
# Fix: split into a reference set (build tier medians) and a scoring set
# (apply medians, never recomputed on itself) using client_hash_id groups
# so no client's rows leak into their own baseline.

import numpy as np

clients = feature_df['client_hash_id'].unique()
rng = np.random.default_rng(42)
rng.shuffle(clients)
split_point = int(len(clients) * 0.5)
reference_clients = set(clients[:split_point])
scoring_clients = set(clients[split_point:])

reference_df = feature_df[feature_df['client_hash_id'].isin(reference_clients)]
scoring_df = feature_df[feature_df['client_hash_id'].isin(scoring_clients)].copy()

safe_tier_medians = reference_df.groupby('position_tier')['ctr'].median()

scoring_df['expected_ctr_safe'] = scoring_df['position_tier'].map(safe_tier_medians)
scoring_df['ctr_gap_safe'] = scoring_df['expected_ctr_safe'] - scoring_df['ctr']

print("Reference clients:", len(reference_clients), "| Scoring clients:", len(scoring_clients))
print(scoring_df[['client_hash_id', 'position_tier', 'ctr', 'expected_ctr_safe', 'ctr_gap_safe']].head())

Reference clients: 20 | Scoring clients: 21
            client_hash_id position_tier    ctr  expected_ctr_safe  \
0  client_62f4a7e64f5e0096        page_1  0.000                0.0   
1  client_62f4a7e64f5e0096        page_1  0.000                0.0   
2  client_62f4a7e64f5e0096        page_1  0.000                0.0   
3  client_62f4a7e64f5e0096        page_1  0.016                0.0   
4  client_62f4a7e64f5e0096        page_1  0.000                0.0   

   ctr_gap_safe  
0         0.000  
1         0.000  
2         0.000  
3        -0.016  
4         0.000  


In [13]:
# Test 2: Does report_date's window bleed past the point I'm claiming to predict at?
print(feature_df['report_date'].unique())
# All rows are month=2026-03, a single closed historical month.
# This means: I am NOT simulating a live "predict forward" scenario —
# I'm scoring a completed window. If I later want to claim forward-looking
# prediction, I'd need month N features -> month N+1 label, which I don't have here.

<DatetimeArray>
['2026-03-01 00:00:00', '2026-03-02 00:00:00', '2026-03-03 00:00:00',
 '2026-03-04 00:00:00', '2026-03-05 00:00:00', '2026-03-06 00:00:00',
 '2026-03-07 00:00:00', '2026-03-08 00:00:00', '2026-03-09 00:00:00',
 '2026-03-10 00:00:00', '2026-03-12 00:00:00', '2026-03-13 00:00:00',
 '2026-03-11 00:00:00', '2026-03-14 00:00:00', '2026-03-15 00:00:00',
 '2026-03-16 00:00:00', '2026-03-17 00:00:00', '2026-03-18 00:00:00',
 '2026-03-19 00:00:00', '2026-03-20 00:00:00', '2026-03-21 00:00:00',
 '2026-03-22 00:00:00', '2026-03-23 00:00:00', '2026-03-24 00:00:00',
 '2026-03-25 00:00:00', '2026-03-26 00:00:00', '2026-03-27 00:00:00',
 '2026-03-28 00:00:00', '2026-03-30 00:00:00', '2026-03-31 00:00:00',
 '2026-03-29 00:00:00']
Length: 31, dtype: datetime64[us]


In [14]:
# Test 3: Am I pulling any pre-computed action/health flags from the warehouse?
cols_used = feature_df.columns.tolist()
banned_cols = ['health_score', 'needs_indexing', 'is_quick_win',
               'needs_ctr_fix', 'needs_engagement_fix', 'ai_opportunity']
overlap = [c for c in cols_used if c in banned_cols]
print("Banned product-flag columns present:", overlap)
# Expected: [] — none of these exist in fact_content_daily_performance,
# they only appear in the separate FlyRank/internship-lanes example dataset.

Banned product-flag columns present: []


## 4. What I excluded and why

**What I excluded and why**

- **health_score** — pre-computed composite score in the example lane dataset. Using it as a feature would mean the model learns to reproduce an existing score rather than learning from raw signals.

- **needs_indexing, is_quick_win, needs_ctr_fix, needs_engagement_fix** — pre-computed action flags. These are essentially pre-baked answers to the question my capstone is trying to answer itself; including them would be circular.

- **ai_opportunity** — same reasoning: a pre-computed opportunity flag, not a raw signal.

- **ctr, expected_ctr** — components of my own label (ctr_gap). Excluded from model inputs; only used to construct the target.

- **client_hash_id, content_hash_id** — kept as identifiers/join keys only, never as model features (high-cardinality IDs would let the model memorize individual clients/pages instead of generalizing).

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.